In [1]:
import pandas as pd
import math

In [2]:
def year_fraction(start_date, end_date, dc_conv):
    days = (end_date - start_date).days
    if dc_conv == "ACT/365.25":
        return days/365.25
    if dc_conv == "ACT/365":
        return days/365
    if dc_conv == "ACT/360":
        return days/360
    elif dc_conv == "ACT/ACT":
        if days == 0:
            return 0.0

        total = 0.0
        current = start_date

        while current < end_date:
            next_year = pd.Timestamp(
                year=current.year + 1,
                month=1,
                day=1
            )

            period_end = min(end_date, next_year)
            days_in_year = 366 if current.is_leap_year else 365

            total += (period_end - current).days / days_in_year
            current = period_end

        return total
    else:
        raise ValueError("day_count must be ACT/ACT, ACT/365.25, ACT/365, or ACT/360")
    

In [5]:
def DF(t, df):
    if t in df.keys():
        return df[t]
    t_vals = sorted(df.keys())
    if t < t_vals[0]:
        return df[t_vals[0]]
    elif t > t_vals[-1]:
        return df[t_vals[-1]]
    for idx in range(len(df) - 1):
        t1 = t_vals[idx] 
        t2 = t_vals[idx + 1]
        if t1 < t and t2 > t:
            d = log_interpolate(t, df[t1], df[t2], t1, t2)
            return d

In [7]:
def log_interpolate(t, d1, d2, t1, t2):
    log_d1 = math.log(d1)
    log_d2 = math.log(d2)
    log_d = log_d1 + ((log_d2 - log_d1)/(t2 - t1))*(t - t1)
    d = math.exp(log_d)
    return d
    

In [9]:
def bootstrap_zero_from_par(par_yields_pct, frequency = 2, dc_conv = "ACT/ACT"):

    
    time_to_maturities = list(par_yields_pct.columns)
    res = []
    
    for curve_date in par_yields_pct.index:
        df = {}
        r = {}
        price = 100
        face_value = 100
        cf_period = 12//frequency
        start_date = curve_date
        
        for T in time_to_maturities:
            months_to_maturity = int(round(float(T) * 12))
            end_date = start_date + pd.DateOffset(months = months_to_maturity)
            T_yf = year_fraction(start_date, end_date, dc_conv)
            
            # Deposits
            if months_to_maturity < cf_period:
                rt = par_yields_pct.loc[curve_date, T]/100 # Zero rate for short-tenor bond
                r[T] = rt
                df[T] = (1 + rt/frequency)**(-1*(frequency*T_yf))
                continue
            
            c = par_yields_pct.loc[curve_date, T]/100 # Percentage to decimal
            
            npv_cf = 0
            final_cf = 0
            date = end_date
            
            while date > start_date:
                yf = year_fraction(start_date, date, dc_conv)
                
                if date == end_date:
                    final_cf = face_value*(1+c/frequency)
                else:
                    cf = face_value*c/frequency
                    npv_cf += cf*DF(yf, df)

                date -= pd.DateOffset(months=cf_period)

            df[T] = (price - npv_cf)/final_cf

            
            r[T] = frequency*((df[T])**(-1/(frequency*T_yf)) - 1)
        
        res.append(r)

    zero_rates = pd.DataFrame(res)
    zero_rates.index = par_yields_pct.index
    zero_rates_pct = zero_rates*100

    return zero_rates_pct
